In [1]:
import sys
try:
    import pyLOM
except ImportError as e:
    print(f'Error importing pyLOM: {e}')
    print('Importing with local repository')
    sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO, SAM

def read_db_CODA(datafolder, case_idx):
    db = FRODO(root_dir = datafolder, format = 'CODA', initial_parse = True)
    
    for id, type, var_excluded in zip([3, 4], ['surface', 'volume'], [[f'BoundaryValues_CoefSkinFriction{coord}' for coord in ['X', 'Y', 'Z']], []]):
        db.extract_inputs(
            id_groups = (id,),
            cases_idx = case_idx,
            vtu_type=type,
            verbose=False
            )

        for stage in [0, 1]:
            db.extract_outputs(
                id_groups=(id,),
                stage=stage, cases_idx = case_idx,
                var_name_excluded = var_excluded,
                vtu_type=type,
                )
    
    db.sets.interpolate_vol2surf(
        vol_group = '4',
        surf_group = '3',
        stage = str(stage),
        vars = 'all',
    )
    
    db.sets.remove_data(id_group='4', stage=None, verbose=False)

    return db

Error importing pyLOM: No module named 'pyLOM'
Importing with local repository


0 Warning! Import - NVTX not present!


In [ ]:
# db_prueba = read_db_CODA(
#     datafolder = '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_propose_0',
#     case_idx = list(range(3))
# )
# db_prueba.summary_data()

In [2]:
from typing import Union
import os
def juntar_db(list_folders:Union[tuple[str], list[str]], case_idx:Union[tuple, list], name: str = None):
    list_db = []
    
    for folder, cases in zip(list_folders, case_idx):
        db = read_db_CODA(datafolder = folder, case_idx=cases)
        
        if db.data_dict['CADGroup_3']['FlCc'].shape[-1] > 2:
            flcc = db.data_dict['CADGroup_3']['FlCc'][:, :-1]
            design_vars = db.metadata['design_vars']
            db.metadata['design_vars'] = design_vars[:-1]
            db.data_dict['CADGroup_3']['FlCc'] = flcc
    
        list_db.append(db)
        
    db_full = FRODO.merge_datasets(
        root_dir='/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_completed',
        name = name,
        sources = [(db, '3') for db in list_db], #[(db_0, '3'), (db_1, '3'), (db_trans, '3')],
        new_group_id='3_completo',
        k=4,
        mesh_ref=0,
        cache=True,
        get_df_metrics_attr={
            'var_metrics': ['CoefLift', 'CoefDrag', 'CoefMomentY'],
            'iter_var': 1000,
            'save' : False
        }
        
    )
    
    return db_full

isTest = True

if isTest:
    case_idx = list(range(3))
    tuple_cases = (case_idx, 'all', case_idx, case_idx)
    
else:
    # Base de datos original
    case_idx = list(range(100))
    fuera = [64, 79, 87, 88, 94]
    for c in fuera:
        case_idx.remove(c)

    tuple_cases = (case_idx, 'all', 'all', 'all') # original, rest, transonic, propose_0
    
db_completo = juntar_db(
    list_folders = [
        os.path.join('/home/m.jaraiz/Documentos/DATASETS/data_TIFON/', folder) for folder in ['rans3_basic', 'rans3_basic_rest', 'rans3_transonic_1', 'rans3_propose_0']
        ],
    case_idx=tuple_cases,
    name = "complete"
)


 NEW CODA SIMULATION WILL BE LOADED FROM /home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_basic
100 simulations found.
Parse took: 0.0777 s

 NEW CODA SIMULATION WILL BE LOADED FROM /home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_basic_rest
5 simulations found.
Parse took: 0.0061 s

 NEW CODA SIMULATION WILL BE LOADED FROM /home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_1
80 simulations found.
Parse took: 0.0594 s

 NEW CODA SIMULATION WILL BE LOADED FROM /home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_propose_0
100 simulations found.
Parse took: 0.0841 s
['total_iterations_stage0', 'aoa', 'mach'] ['aoa', 'mach', 'stage', 'case_idx', 'h', 're', 'folder', 'CoefLift_mean_stage0', 'CoefLift_var_stage0', 'CoefDrag_mean_stage0', 'CoefDrag_var_stage0', 'CoefMomentY_mean_stage0', 'CoefMomentY_var_stage0', 'CoefLift_mean_stage1', 'CoefLift_var_stage1', 'CoefDrag_mean_stage1', 'CoefDrag_var_stage1', 'CoefMomentY_mean_stage1', 'CoefMomentY_var_stage1'] ['aoa', 'mach']
[

In [ ]:
path = '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_completed/outputs'
db_completo.sets.create_NN_pylom(id_groups = '3_completo', stage='0', save_path = path)
db_completo.sets.create_NN_pylom(id_groups = '3_completo', stage='1', save_path = path)